# 001 — Data Wrangling: Consolidación de Indicadores ODS de México

Este notebook carga, limpia y consolida los **333 indicadores** de los Objetivos de Desarrollo
Sostenible (ODS) de México disponibles en la carpeta `../data/`.

**Objetivo**: Crear un DataFrame unificado `Periodo × Indicadores` con valores nacionales
para análisis de correlación posterior.

## 1. Importar librerías y configurar rutas

In [1]:
import pandas as pd
import numpy as np
import os
import re
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data'
OUTPUT_DIR = '../data/consolidated'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Directorio de datos: {os.path.abspath(DATA_DIR)}")

Directorio de datos: /home/gusgus/HackODS2026/ODSiosos_HackODS_2026/data


## 2. Catálogo de indicadores
Cargar el mapeo de códigos de indicadores a sus descripciones completas.

In [2]:
# El catálogo está repetido en cada carpeta, usamos el primero que encontremos
catalog_path = None
for folder in sorted(os.listdir(DATA_DIR)):
    cat_dir = os.path.join(DATA_DIR, folder, 'catalogos')
    if os.path.isdir(cat_dir):
        for f in os.listdir(cat_dir):
            if f == 'claves_codigos_indicadores_es.csv':
                catalog_path = os.path.join(cat_dir, f)
                break
    if catalog_path:
        break

catalog = pd.read_csv(catalog_path, encoding='utf-8')
catalog.columns = ['codigo', 'descripcion']
# Limpiar HTML tags de las descripciones
catalog['descripcion'] = catalog['descripcion'].str.replace(r'<[^>]+>', '', regex=True).str.strip()
print(f"Catálogo cargado: {len(catalog)} indicadores")
catalog.head(10)

Catálogo cargado: 260 indicadores


,codigo,descripcion
0,1.1.1,Proporción de la población que vive por debajo...
1,1.1.1.a,Proporción de la población que vive por debajo...
2,1r.1.1,Porcentaje de la población que vive por debajo...
3,1.2.1(1),Proporción de la población que vive por debajo...
4,1.2.1(2),Proporción de la población que vive por debajo...
5,1.2.1.a,Proporción de la población que vive por debajo...
6,1.2.2(1),"Proporción de hombres, mujeres y niños de toda..."
7,1.2.2(2),"Proporción de hombres, mujeres y niños de toda..."
8,1.2.2.a,"Proporción de hombres, mujeres y niños de toda..."
9,1.4.2,Proporción del total de la población adulta co...


## 3. Extraer metadatos de cada indicador

De cada archivo `*_met_es.txt` extraemos: nombre, ODS (Goal), Meta (Target), 
tipo de indicador y unidad de medida.

In [3]:
def parse_metadata(met_path):
    """Extrae campos clave del archivo de metadatos."""
    info = {}
    try:
        with open(met_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Extraer campos con regex
        patterns = {
            'title': r'Title:\s*(.+)',
            'unit': r'Unit of measure:\s*(.+)',
            'goal': r'Goal:\s*(.+)',
            'target': r'Target:\s*(.+)',
            'type': r'Type of indicator:\s*(.+)',
            'temporal': r'Temporal:\s*(.+)',
            'spatial': r'Spatial:\s*(.+)',
        }
        for key, pattern in patterns.items():
            match = re.search(pattern, content)
            info[key] = match.group(1).strip() if match else None
    except Exception as e:
        pass
    return info

# Recopilar metadatos de todas las carpetas
metadata_list = []
indicator_folders = sorted([d for d in os.listdir(DATA_DIR) 
                            if os.path.isdir(os.path.join(DATA_DIR, d)) and d != 'consolidated'])

for folder in indicator_folders:
    met_dir = os.path.join(DATA_DIR, folder, 'metadatos')
    if not os.path.isdir(met_dir):
        continue
    for f in os.listdir(met_dir):
        if f.endswith('_met_es.txt'):
            met_path = os.path.join(met_dir, f)
            info = parse_metadata(met_path)
            # Extraer el código del indicador del nombre de la carpeta
            indicator_code = folder.replace('_es', '')
            info['indicator_id'] = indicator_code
            info['folder'] = folder
            metadata_list.append(info)

metadata_df = pd.DataFrame(metadata_list)

# Extraer número de ODS del Goal
def extract_ods_number(goal_text):
    if pd.isna(goal_text):
        return None
    match = re.match(r'(\d+)', str(goal_text).strip())
    return int(match.group(1)) if match else None

metadata_df['ods_number'] = metadata_df['goal'].apply(extract_ods_number)
print(f"Metadatos extraídos para {len(metadata_df)} indicadores")
print(f"\nDistribución por ODS:")
print(metadata_df['ods_number'].value_counts().sort_index())

Metadatos extraídos para 333 indicadores

Distribución por ODS:
ods_number
1     31
2     17
3     42
4     33
5     29
6     25
7      6
8     31
9     16
10    11
11    16
12     8
13     7
14     4
15     9
16    36
17    12
Name: count, dtype: int64


In [4]:
metadata_df[['indicator_id', 'title', 'ods_number', 'unit']].head(20)

,indicator_id,title,ods_number,unit
0,1.1.1.A,1.1.1.a Proporción de la población que vive po...,1,Porcentaje
1,1.1.1,1.1.1 Proporción de la población que vive por ...,1,Porcentaje
2,1.2.1(1),1.2.1(1) Proporción de la población que vive p...,1,Porcentaje
3,1.2.1(2),1.2.1(2) Proporción de la población que vive p...,1,Porcentaje
4,1.2.1.A,1.2.1.a Proporción de la población que vive po...,1,Porcentaje
5,1.2.1.B,1.2.1.b Proporción de la población que vive po...,1,Porcentaje
6,1.2.2(1),"1.2.2(1) Proporción de hombres, mujeres y niño...",1,Porcentaje
7,1.2.2(2),"1.2.2(2) Proporción de hombres, mujeres y niño...",1,Porcentaje
8,1.2.2.A,"1.2.2.a Proporción de hombres, mujeres y niños...",1,Porcentaje
9,1.2.2.B,"1.2.2.b Proporción de hombres, mujeres y niños...",1,Porcentaje


## 4. Carga y normalización de los datos

Cada carpeta contiene un archivo `*_sh_es.csv` con la serie de tiempo principal.
Detectamos si es formato **nacional** o **estatal** y extraemos el valor nacional.

In [5]:
def load_indicator_data(folder_path, indicator_id):
    """
    Carga el archivo _sh_es.csv de un indicador y extrae la serie temporal nacional.
    
    Retorna un DataFrame con columnas: [periodo, valor]
    """
    data_dir = os.path.join(folder_path, 'conjunto_de_datos')
    if not os.path.isdir(data_dir):
        return None
    
    # Buscar archivo _sh_es.csv
    sh_files = [f for f in os.listdir(data_dir) if f.endswith('_sh_es.csv')]
    if not sh_files:
        return None
    
    sh_path = os.path.join(data_dir, sh_files[0])
    
    try:
        df = pd.read_csv(sh_path, encoding='utf-8', dtype=str)
        # Limpiar nombres de columnas
        df.columns = [c.strip().strip('"') for c in df.columns]
        
        # Detectar formato: estatal tiene 'cvegeo' o 'Entidad_federativa'
        is_state_level = any(col in df.columns for col in ['cvegeo', 'Entidad_federativa'])
        
        if is_state_level:
            # Filtrar solo el registro nacional (cvegeo == 00)
            if 'cvegeo' in df.columns:
                df['cvegeo'] = df['cvegeo'].astype(str).str.strip().str.zfill(2)
                df = df[df['cvegeo'] == '00'].copy()
            elif 'Entidad_federativa' in df.columns:
                df = df[df['Entidad_federativa'].str.contains('Estados Unidos Mexicanos', na=False)].copy()
            
            if df.empty:
                return None
            
            # El valor está en la última columna numérica (o columna que no sea Periodo/cvegeo/Entidad)
            exclude_cols = {'Periodo', 'cvegeo', 'Entidad_federativa', 'Serie', ''}
            value_cols = [c for c in df.columns if c not in exclude_cols and c.strip() != '']
            
            if not value_cols:
                return None
            
            value_col = value_cols[0]  # Tomar la primera columna de valor
            result = df[['Periodo', value_col]].copy()
            result.columns = ['periodo', 'valor']
        
        else:
            # Formato nacional: buscar columna Periodo y columnas de valores
            if 'Periodo' not in df.columns:
                # A veces la primera columna es la Serie
                return None
            
            exclude_cols = {'Serie', 'Periodo', ''}
            value_cols = [c for c in df.columns if c not in exclude_cols and c.strip() != '']
            
            if not value_cols:
                return None
            
            # Si hay una columna "Total" o similar, preferirla
            total_col = None
            for c in value_cols:
                if 'Total' in c or 'total' in c:
                    total_col = c
                    break
            
            if total_col is None:
                total_col = value_cols[0]
            
            result = df[['Periodo', total_col]].copy()
            result.columns = ['periodo', 'valor']
        
        # Limpiar valores
        result['periodo'] = pd.to_numeric(result['periodo'].str.strip().str.strip('"'), errors='coerce')
        result['valor'] = result['valor'].str.strip().str.strip('"').str.replace(',', '')
        result['valor'] = pd.to_numeric(result['valor'], errors='coerce')
        result = result.dropna(subset=['periodo', 'valor'])
        result['periodo'] = result['periodo'].astype(int)
        result['indicator_id'] = indicator_id
        
        return result
    
    except Exception as e:
        return None

In [6]:
# Cargar todos los indicadores
all_data = []
load_errors = []

for folder in indicator_folders:
    indicator_id = folder.replace('_es', '')
    folder_path = os.path.join(DATA_DIR, folder)
    
    df = load_indicator_data(folder_path, indicator_id)
    if df is not None and len(df) > 0:
        all_data.append(df)
    else:
        load_errors.append(indicator_id)

print(f"✅ Indicadores cargados exitosamente: {len(all_data)}")
print(f"⚠️  Indicadores sin datos nacionales: {len(load_errors)}")

# Combinar todos los datos
all_indicators_df = pd.concat(all_data, ignore_index=True)
print(f"\nRegistros totales: {len(all_indicators_df):,}")
print(f"Periodos únicos: {sorted(all_indicators_df['periodo'].unique())[:5]}...{sorted(all_indicators_df['periodo'].unique())[-5:]}")
print(f"Indicadores únicos: {all_indicators_df['indicator_id'].nunique()}")

✅ Indicadores cargados exitosamente: 292
⚠️  Indicadores sin datos nacionales: 41

Registros totales: 3,611
Periodos únicos: [np.int64(1970), np.int64(1971), np.int64(1972), np.int64(1973), np.int64(1974)]...[np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
Indicadores únicos: 292


## 5. Construir tabla pivote: Periodo × Indicadores

In [7]:
# Pivotear los datos: filas = Periodo, columnas = Indicador, valores = valor
# Si hay duplicados (mismo indicador/año), tomar el promedio
pivot_df = all_indicators_df.pivot_table(
    index='periodo', 
    columns='indicator_id', 
    values='valor', 
    aggfunc='mean'
)

# Ordenar por año
pivot_df = pivot_df.sort_index()

print(f"Tabla pivote: {pivot_df.shape[0]} años × {pivot_df.shape[1]} indicadores")
print(f"\nCobertura temporal: {pivot_df.index.min()} - {pivot_df.index.max()}")

# Estadísticas de cobertura
coverage = pivot_df.notna().sum()
print(f"\nIndicadores con ≥5 observaciones: {(coverage >= 5).sum()}")
print(f"Indicadores con ≥10 observaciones: {(coverage >= 10).sum()}")
print(f"Indicadores con ≥15 observaciones: {(coverage >= 15).sum()}")

pivot_df.head()

Tabla pivote: 52 años × 292 indicadores

Cobertura temporal: 1970 - 2026

Indicadores con ≥5 observaciones: 199
Indicadores con ≥10 observaciones: 121
Indicadores con ≥15 observaciones: 90


indicator_id,1.1.1,1.2.1(1),1.2.1(2),1.2.1.A,1.2.2(1),1.2.2(2),1.3.1.,1.4.1,1.4.2,1.5.1,...,9.2.2(2),9.3.1,9.3.2,9.4.1,9.5.1,9.5.2,9.B.1,9.C.1,9N.2.1,9N.3.1
periodo,,,,,,,,,,,,,,,,,,,,,
1970,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1971,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1972,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1973,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1974,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 6. Resumen de indicadores cargados por ODS

In [8]:
# Enriquecer con metadatos
indicator_summary = metadata_df[['indicator_id', 'title', 'ods_number', 'unit']].copy()
indicator_summary['loaded'] = indicator_summary['indicator_id'].isin(pivot_df.columns)
indicator_summary['n_observations'] = indicator_summary['indicator_id'].apply(
    lambda x: pivot_df[x].notna().sum() if x in pivot_df.columns else 0
)

# Resumen por ODS
ods_summary = indicator_summary.groupby('ods_number').agg(
    total_indicators=('indicator_id', 'count'),
    loaded_indicators=('loaded', 'sum'),
    avg_observations=('n_observations', 'mean')
).reset_index()
ods_summary.columns = ['ODS', 'Total', 'Cargados', 'Obs. promedio']

ods_names = {
    1: 'Fin de la Pobreza', 2: 'Hambre Cero', 3: 'Salud y Bienestar',
    4: 'Educación de Calidad', 5: 'Igualdad de Género', 6: 'Agua Limpia',
    7: 'Energía Asequible', 8: 'Trabajo Decente', 9: 'Industria e Innovación',
    10: 'Reducción de Desigualdades', 11: 'Ciudades Sostenibles', 12: 'Producción Responsable',
    13: 'Acción por el Clima', 14: 'Vida Submarina', 15: 'Vida de Ecosistemas Terrestres',
    16: 'Paz, Justicia e Instituciones', 17: 'Alianzas para los Objetivos'
}
ods_summary['Nombre ODS'] = ods_summary['ODS'].map(ods_names)
print("Resumen de indicadores por ODS:")
ods_summary

Resumen de indicadores por ODS:


,ODS,Total,Cargados,Obs. promedio,Nombre ODS
0,1,31,18,5.903226,Fin de la Pobreza
1,2,17,12,4.941176,Hambre Cero
2,3,42,37,18.214286,Salud y Bienestar
3,4,33,27,8.424242,Educación de Calidad
4,5,29,27,3.379310,Igualdad de Género
5,6,25,25,5.720000,Agua Limpia
6,7,6,6,15.333333,Energía Asequible
7,8,31,29,12.580645,Trabajo Decente
8,9,16,16,14.375000,Industria e Innovación
9,10,11,9,4.727273,Reducción de Desigualdades


## 7. Exportar datos consolidados

In [9]:
# Guardar la tabla pivote
output_path = os.path.join(OUTPUT_DIR, 'indicadores_nacionales.csv')
pivot_df.to_csv(output_path, encoding='utf-8')
print(f"✅ Datos consolidados guardados en: {output_path}")
print(f"   Dimensiones: {pivot_df.shape[0]} años × {pivot_df.shape[1]} indicadores")

# Guardar metadatos
metadata_output = os.path.join(OUTPUT_DIR, 'metadatos_indicadores.csv')
indicator_summary.to_csv(metadata_output, index=False, encoding='utf-8')
print(f"✅ Metadatos guardados en: {metadata_output}")

# Guardar los datos en formato largo también (útil para visualizaciones)
long_output = os.path.join(OUTPUT_DIR, 'indicadores_largo.csv')
all_indicators_enriched = all_indicators_df.merge(
    metadata_df[['indicator_id', 'title', 'ods_number', 'unit']], 
    on='indicator_id', 
    how='left'
)
all_indicators_enriched.to_csv(long_output, index=False, encoding='utf-8')
print(f"✅ Formato largo guardado en: {long_output}")

✅ Datos consolidados guardados en: ../data/consolidated/indicadores_nacionales.csv
   Dimensiones: 52 años × 292 indicadores
✅ Metadatos guardados en: ../data/consolidated/metadatos_indicadores.csv
✅ Formato largo guardado en: ../data/consolidated/indicadores_largo.csv


In [10]:
print("\n📊 Resumen final:")
print(f"   - Indicadores procesados: {pivot_df.shape[1]}")
print(f"   - Rango temporal: {pivot_df.index.min()} - {pivot_df.index.max()}")
print(f"   - Archivos generados:")
print(f"     • {output_path}")
print(f"     • {metadata_output}")
print(f"     • {long_output}")


📊 Resumen final:
   - Indicadores procesados: 292
   - Rango temporal: 1970 - 2026
   - Archivos generados:
     • ../data/consolidated/indicadores_nacionales.csv
     • ../data/consolidated/metadatos_indicadores.csv
     • ../data/consolidated/indicadores_largo.csv
